## Using MLflow for Experiment Tracking
### An introduction to MLflow
#### Chellenges of Machine Learning Lifecycle
* Tracking
  + Experimentation and models are hard to track and compare including models and model metrics
* reproducibility
  + difficult to reproduce code and results across different deployment platforms, environments, and user workspace
* deployment
  + models are hard to deploy. Too many options and no standards
#### What is MLflow?
* attempts to simplify and automate machine learning lifecycle
* open-source platform used to help users manage the ML lifecycle by providing
  + experiment tracking
  + reproducibility
  + deployment
  + centralized model registry
#### Components of MLflow
* MLflow tracking
  + record metrics and parameters from training runs (postgreSQL)
  + Query data from experiments
  + Store models, artifacts and code
* MLflow models
  + standarize the way models are packaged to enable a more streamlined deployment process
  + provides a way to build customization such as model loading and model inference
* Model Registry
  + store and version ML modles with tags for certain model deployment environments
  + load and deploy ML models
* MLflow projects
  + standardize packaging ML code so that it can be reproduced across a variety of environments, platforms and user workspaces
  + package ML code for repeatability to aid in automation
#### MLflow Experiments
* An experiment is a way that MLflow organizes tracking of model training runs
* when we begin tracking, we must specify which experiment to use so that we know where to find that data later
* Working with experiments
  + by MLflow client as a low level API for interacting directly with MLflow
  + by MLflow module as a high level API used for starting and directly managing current training runs
  + MLflow module has an extra function of set\_experiment to set experiment of the current training runs, which allows us to track these runs
```Python
import mlflow
# create Experiments by client and mlflow module
client.create_experiment("Name")
mlflow.create_experiment("Name")

# tag Experiments by client and mlflow module
client.set_experiment_tag("Name", k, v)
mlflow.set_experiment_tag(k, v)

# delete Experiments by client and mlflow module
client.delete_experiment("Name")
mlflow.delete_experiment("Name")

# set Experiment of the current run
mlflow.set_experiment("Name")
```
  + The following is a code example to use mlflow module to track the crrent run by experiments
```Python
import mlflow

# create new Experiment
mlflow.create_experiment("My Experiment)

# Tag new experiment so that we can use the tag for better search later
mlflow.set_experiment_tag("scikit-learn", "lr")

# We must set the experiment for the following runs so MLflow knows where to place data
mlflow.set_experiment("My Experiment")
```

#### MLflow Tracking
* why tracking?
  + to help model engineering and evaluation processes
  + most models are built and reiterated several times to ensure the best performance
  + each time when a model is trained, we track performance metrics, code, parameters and other artifacts during training
* What to track?
  + model metrics (F1, Recall, Accuracy, MSE, etc.)
    + use `log_metric("accuracy", 0.99)` for single metric
    + use `log_metrics({"accuracy": 0.99, "loss": 0.50}) for multiple metrics
  + parameters (library specific) 
    + use `log_param("n_jobs", 1)` for single metric
    + use `log_params({"n_jobs": 1, "fit_intercept": False}) for multiple metrics
  + code (train.py)
  + other artifacts (tokenizer, pickle, etc.)
    + use `log_artifact("file.py") for a single file
    + use `log_artifacts("./directory/") for multiple files in a directory    
* Training runs
  + each run corresponds to a new model training
  + A run is placed within an experiment
  + invoked via `mlflow.start_run()`
    + return value of start_run() is a run object
  + mlflow module will continue logging to the active run until the code exits or the `mlflow.end_run()` is called
* How to track?
  + code example for tracking model training
```Python
import mlflow

# create new Experiment
mlflow.create_experiment("My Experiment)

# Tag new experiment so that we can use the tag for better search later
mlflow.set_experiment_tag("scikit-learn", "lr")

# We must set the experiment for the following runs so MLflow knows where to place data
mlflow.set_experiment("My Experiment")

# start a model training as a new run
run = mlflow.start_run()
run.info

# model training code here
lr = LogisticRegression(n_jobs=1)

# model evaluation code
lr.fit(X, y)
score = lr.score(X, y)

# log a metric
mlflow.log_metric("score", score)

# log a parameter
mlflow.log_param("n_jobs", 1)

# log an artifact
mlflow.log_artifact("train_code.py")
```
* search runs
  + we can define max_results for search
  + order_by to define columns to sort in ASC or DESC
  + filter_string
  + experiment_names: name(s) of experiments to query
  + code example of searching/filtering runs
  + default output is pandas dataframe that organize all run inforamtion including run_id, experiment_id, tags, metrics and paramters in columns and runs as rows
```Python
import mlflow

# filter string
f1_score_filter = "metrics.f1_score > 0.69"

# search runs
mlflow.search_runs(experiment_names=["Insurance Experiment"],
                   filter_string=f1_score_filter,
                   order_by=["metrics.precision_score DESC"]
                  )
```             


### MLflow Models
#### why use MLflow models?
* standardize packaging ML models
* standarizing models allows for easy integration between popular ML libraries and deployment tools.
* packaging models refers to the process of placing all application files and resources in a strategic way that can be distributed more easily
* standards are defined in a format convention used for saving models in differnt model "Flavors" that can be understood by different downstream tools
* MLflow packages models using a standardized storage format and folder structure for all MLflow models, including MLmodel file (a YAML file), model.pkl, python_env.yaml and requirments.txt
  + MLmodel file is a yaml file that defines important information about the model
    + defines information to load the model
    + contains virtual environment, python version and scikit-learn version used to build the model
#### using flavors
* makes it possible to write tools to work with most popular ML libraries using MLflow
  + Python function
  + R function
  + H2O
  + Keras
  + MLeap
  + PyTorch
  + Scikit-learn
  + Spark MLib
  + XGBoost  
* simplify the process of logging, packaging, and loading models and minimizing the need to write custom code
* some of MLflow's built-in flavors support autologging to handle automatic logging of metrics, params and models to MLflow traing w/o the need for defining explicit logging statements
* scikit-learn flavor example
```Python
import mlflow
from sklearn.linear_model import LinearRegression

# use auto-logging by sklearn flavor
mlflow.sklearn.autolog()

# train the model and model will be automatically logged on model.fit()
lr = LinearRegression()
lr.fit(X, y)
```


#### Model API
* Model API is used to interact with models that allows you to save, log and load MLflow models using a flavor
  ```YAML
  # save a model to the local filesystems. support both relative and absolute local path
  mlflow.sklearn.save_model(model, local_folder_path)

  # log a model as an artifact to MLflow tracking. support runs:/<mlflow_run_id>/run-relative/path/to/model
  mlflow.sklearn.log_model(model, artifact_folder_path)

  # load a model from local filesystem or from MLflow Tracking. support S3 as s3://my_bucket/path/to/model
  mlflow.sklearn.load_model(model_url)

  # instead of using autolog, we can explicitly log model to tracking system by log_model
  # this allows us to define model name and artifacts path to save the model. MLflow will
  # save the entire model folder as specified by tracking_path
  mlflow.sklearn.log_model(lr, "tracking_path")

  # when we want to load the model of the last run, we can get the run_id of last active run
  # and then use the run_id from run.info to retrieve the model
  run = mlflow.last_active_run()
  run_id = run.info.run_id
  model = mlflow.sklearn.load_model(f"runs:/{run_id}/tracking_path")
  ```
* Custom models
  + built-in flavor as python_function (mlflow.pyfunc)
    + supports `save_model()`, `log_model()` and `load_model()`
  + to create a custom model class
    + we define our custom model class that inherit mlflow.pyfunc.PythonModel
    + we then implement the two functions:
      + `load_context()`: loads artifacts when `mlflow.pyfunc.load_model()` is called
      + `predict_()`: takes model input and performs user defined evaluation
  + code example
```Python
import mlflow.pyfunc

# define the model class
class CustomPredict(mlflow.pyfunc.PythonModel):
    # load artifacts
    def load_context(self, context):
        self.model = mlflow.sklearn.load_model(context.artifacts["cusotm_model"])

    # evaluate input using custom_function()
    def predict(self, context, model_input):
        prediction = self.model.predict(model_input)
        return custom_function(prediction)

# save CustomPredict model to local filesystem using mlflow.pyfunc flavor
mlflow.pyfunc.save_model(path="custom_model", python_model=CustomPredict()) 
mlflow.pyfunc.log_model(airfact_path="custom_model", python_model=CustomPredict())

# load model from local and mlflow tracing system
mlflow.pyfunc.load_model("local")
mlflow.pyfunc.load_model(f"runs/{run_id}/tracking_path")
```

* model evaluation
  + mlflow provide evalution function that can load a trained model and then apply to test dataset to calcualte the model performance
  + estimats of performance and shap values will be generated
  + code example
```Python
# get training and testing Data
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=0)

# train model
lr = LinearRegression()
lr.fit(X_train, y_train)

# build test dataset with feature matrix and target variable in pandas dataframe
# set label column as 'test_label'
eval_data = X_test
eval_data["test_label"] = y_test

# evaluate model with test dataset
mlflow.evaluate(
    f"runs:/{run_id}/model",
    eval_data,
    targets="test_label",
    model_type="regression"
    )
```  

#### Model Serving
* how to use MLflow for model deployment
* serve models as RestAPI with the following endpoints
  + /ping: for health checks
  + /health: for health checks
  + /version: getting the version of MLflow
  + /invocations: for model scoring /inference
  + use port 5000 as default
* model inference using /invocations endpoint
  + accepts both csv and json as input
  + specify content-type as application/json or application/csv
  + for csv format, it must be a valid pandas data frame and convert using pandas_df.to_csv() method
  + for json format, it must be in dataframe_split (recommended) or dataframe_records format
    + ```JSON
      {
       "dataframe_split": {
          "columns": ["sex", "age", "weight"],
          "data": [["male", 23, 160], ["female", 33, 120]]
          }
      }
      ```
* to serve a model, we use `mlflow models serve` command line
  + first, use `mlflow models serve -m runs:/run_id/model_path` to load model. you can also use a local path or S3 path to load model
  + then use invocations endpoint to get inference
    ```bash
    curl http://127.0.0.1:5000/invocations -H 'Content-Type: application/json' -d '{
      "dataframe_split": {
         "columns": ["sex", "age", "weight"],
         "data": [["male", 23, 160], ["female", 33, 120]]
    }
    }
    ```

### Model Registry
* managing lifecycle of model through different software environments such as development, staging and production
* MLflow model registry provides access to models for collaboration through a UI and MLflow Client module. It also provides a way to manage the lifecycle of models through model versioning and model staging
* MLflow model registry consists of four major concepts:
  + Model: an MLflow model that is created and logged to MLflow tracking through a run or experiment, which is eligible to be registered with Mlflow model registry
  + registered model: once a model is registered, the model then obtain a version and assigned a stage
    + model version: version increments with each new registered model using the same model name
    + model stage: can be assigned one of None, Staging, Production and Archived
#### Interacting with model registry
* interacting with model registry can be done by either MLflow Client, or UI
* we use `client.create_registered_model(name=<model_name>)` to create an area in model registry to register versioned models at a later time
* after registering models with different names and versions, we can search these models using `search_registered_models(filter_string)` method
  ```Python
  from mlflow import MlflowClinet

  client = MlflowClient()
  client.create_registered_model(name="Unicorn") # pass the name of our model for registry

  # search registered model similar to search models using Client
  client.search_registered_models(fiter_string=MY_FILTER_STRING)
  ```
* search registered models using filter_string by defining
  + identifiers
    + name of model
    + tags associated with model
  + comparators
    + equal to: =
    + not equal to: !=
    + LIKE: case-sensitive pattern match
    + ILIKE: case-insensitive pattern match
  + example of filter_string:
  ```bash
  unicorn_filter_string = "name LIKE 'Unicorn%'"
  client.search_registered_models(filter_string=unicorn_filter_string)
  ```

#### Registering MLflow models
* registering MLflow models
  + provide model versions
    + follows traditional sofware development to version control models
    + track changes
  + collaboration
    + share models between different roles in teams
    + improve models
* when a model is registered to model registry, MLflow
  + first check if the model exists in the registry, if not, it will create the registered model (here "Unicorn" in the code)
    + you can first create the registered model by `mlflow.create_registered_model(<model_name>)`
  + create a new version starting from 1
* code example
  ```Python
  import mlflow
  import mlflow.sklearn
  from sklearn.linear_model import LogisticRegression 
  from mlflow import MlflowClinet
  
  # for existing model that already has local or tracking uri
  mlflow.register_model("./model", "Unicorn")
  mlflow.register_model("runs:/run_id/model", "Unicorn")

  # for new model
  lr = LogisticRegression()
  lr.fit(X, y)
  
  # register model by mlflow_model, artifact_uri, registered_model_name
  mlflow.sklearn.log_model(lr, "model", registered_model_name="Unicorn")

  # we can search the registered models
  client = MlflowClient()
  filter_string = "name = 'Unicorn'"
  client.search_registered_models(filter_string=filter_string)
  ```
#### Model stages
* In addition to model version control, model registry provides a way to manage model lifecycle through different software environments
* To transition through different software development environments, MLflow provides "Model Stages"
* A model stage represent a phase in the model lifecycle and can be assigned to a registered model version
* each stage can be considered a set of rules defined within an individual organization as to how they handle software development environments
* MLflow provides predefined stages of "None", "Staging", "Production" and "Archived"
* A versioned model can be transitioned from one stage to another but can only have one assigned stage at any given time
* Model stages
  + None: default, which means model has not yet received a stage
  + staging: when a model is going through testing and evaluation
  + production: when a model has passed all tests and is ready to be used in production
  + archived: when a model is no longer in use and should be archived
* transitioning model stages can be done either from Registry UI or by `client.transition_model_version_stage()`
  ```Python
  from mlflow import MlflowClient
  client = MlflowClient()

  # Transition to staging
  client.transition_model_version_stage(
      name="Unicorn",
      version=3,
      stage="Staging"
      )

  # after model passed all tests and evaluations of our new stagging model, 
  # transition model version to Production 
  client.transition_model_version_stage(
      name="Unicorn",
      version=3,
      stage="Production"
      )
  ```

#### Model deployment
* model registry provides a simplified model deployment process using both model versions or model stages
* two ways to deploy models
  + use `mlflow.FLAVOR.load_model(<model_uri>)` to load model and serve, e.g., in fastAPI
  + use `mlflow models serve` to directly serve the model by command line tool
* to specify model deployment from model registry, we specify the model uri
  + model uri consists of prefix `models:/` followed by either model version, or model stage
  + `models:/model_name/version`, or `models:/model_name/stage`
* code example using load_model:
```Python
import mlflow.sklearn

# load Unicorn model in staging
model = mlflow.sklearn.load_model("models:/Unicorn/Staging")

# inference
model.predict(data)
```

* code examplle using mlflow models serve command line
```bash
mlflow models serve -m "models:/Unicorn/Production"

curl http://127.0.0.1:5000/incocations -H 'Content-Type: application/json' -d
{
  "dataframe_split": {
      "columns": ["R&D Spend", "Administration", "Marketing Spend", "State"],
      "data": [["165349.20", 136897.80, 471784.10, 1]]
      }
}
```
  

### MLflow project
* simplifies ML lifecycle by providing a way to organize and run ML code in a reproducible manner
  + includes code used to train and build models, track experiments and register models to model registry
  + projects are used to package code into reusable units that allow for simple collaboration among users
  + projects provide portability to run code in different environments like local machines and in the cloud
  + improve and accelerate productivity
#### what is MLflow project
* at its core, a MLflow project is a directory of files containing ML code that can be stored locally or in a git repo such as github
* MLflow projects use a file called MLproject to describe a project
* fold structure of a MLflow project
* ```
  project/
      MLproject
      train_model.py
      python_env.yaml
      requirements.txt
  ```
* MLproject file is a YAML file specifying serveral properties
  + name: name of the project
  + entry_points
    + command(s) to run
    + .py and .sh file
    + workflows
  + python_env
    + python environment
    + python_env.yaml
  + code example of MLproject file
  ```YAML
  # MLproject
  name: salary_model
  entry_points:
    main:
      parameters:
        parameter_1:
          type: data_type
          default: default_value
      command: "python train_model.py"
  python_env: python_env.yaml
  ```

  ```Python
  # train_model.py

  import mlflow
  import mlflow.sklearn
  import pandas as pd
  form sklearn.linear_model import LinearRegression
  from sklearn.model_selection import train_test_split

  df = pd.read_csv("train_data.csv")
  X = df[["experience", "age", "interview_score"]]
  y = df[["salary"]]

  X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=0)

  # set auto logging for scikit-learn flavor
  mlflow.sklearn.autolog()

  # train model
  lr = LinearRegression()
  lr.fit(X_train, y_train) # this will trigger autolog
  ```

#### run MLflow project
  + we can run MLproject from projects API from mlflow.projects module
    + we provide
      + uri: URI to MLproject file
      + entry_point: entry point to start run from MLproject
      + experiment_name: experiment to track training run
      + env_manager: python environment manager: local or virtualenv
  ```Python
  import mlflow

  # run local project in the current folder
  mlflow.projects.run(uri="./", entry_point='main',
                      experiment_name='Salary Model',
                      env_manager='virtualenv'
                     )
  ```
  + another option is to usen `mlflow run` command line with the following options:
    + --entry-point: URI to MLproject file
    + --experiment-name: experiment to track training run
    + --env-manager: python environment manager: local or virtualenv
    + URI
  ```bash
  mlflow run --entry-point main --experiment-name "Salary Model" --env-manager local ./
  ```

#### Specify parameter
* we can specify paramters of MLproject inside an entry_point in MLproject file
  + type: float, string etc. default to string
  + default: True, False, None
  ```YAML
   # MLproject
  name: salary_model
  entry_points:
    main:
      parameters:
        n_jobs_params:
          type: int
          default: 1
        fit_intercept_param:
          type: bool
          default: True
      command: "python train_model.py {n_jobs_param} {fit_intercept_param}"
  python_env: python_env.yaml
  ```

* we need to modify the train.py file specified by main entry_point to accept the parameters
  ```Python
  # train_model.py

  import mlflow
  import mlflow.sklearn
  import pandas as pd
  form sklearn.linear_model import LinearRegression
  from sklearn.model_selection import train_test_split
  import sys

  df = pd.read_csv("train_data.csv")
  X = df[["experience", "age", "interview_score"]]
  y = df[["salary"]]

  X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.7, random_state=0)

  # set auto logging for scikit-learn flavor
  mlflow.sklearn.autolog()

  n_jobs_param = int(sys.argv[1])
  fit_intercept_param = bool(sys.argv[2])
  
  # train model
  lr = LinearRegression(n_jobs=n_jobs_param, fit_intercept=fit_intercept_param)
  lr.fit(X_train, y_train) # this will trigger autolog
  ```
* to run this project from mlflow.Projects API, we use
  ```Python
  import mlflow

  # run local project in the current folder
  mlflow.projects.run(uri="./", entry_point='main',
                      experiment_name='Salary Model',
                      env_manager='virtualenv',
                      parameters={
                          'n_jobs_param':2,
                          'fit_intercept_param': False
                          }
                     )
  ```  
*  to usen `mlflow run` command line       
 ```bash
  mlflow run --entry-point main --experiment-name "Salary Model" --env-manager local \
  -P n_jobs_param=3 -P fit_intercept_param=True ./
  ```      
  

#### Define workflows
* combine multi-steps together
* the basic idea is the following
  + run first step by `step_1 = mlflow.projects.run(uri='./', entry_point='step_1')`
  + pass the step_1 run object to the second step as parameter
  + we can use the attributes of the returned step_1 object
    + step_1.cancel(): terminate a current run
    + step_1.get_status(): get the status of a run
    + step_1.run_id: get run_id of the run
    + step_1.wait(): wait for the run to finish, return True if the run is succeeded, or False if not
* code example
  ```Python
  import mlflow

  # step 1
  step_1 = mlflow.projects.run(
     uri='./',
     entry_point='step_1'
     )

  step_1_run_id = step_1.run_id

  # step 2
  step_2 = mlflow.projects.run(
      uri='./',
      entry_point='step_2',
      parameters={
          'run_id': step_1_run_id
          }
      )     
 ```    